# Public Portfolio Note

**Purpose:** Document the county-clustered OLS models, model diagnostics, coefficient visualizations, and sensitivity analyses used in the final report.

**Inputs:** Excluded cleaned modeling dataset with constructed percentile gaps, standardized covariates, and institutional subgroup indicators.

**Outputs:** Aggregate model summaries, diagnostics, sensitivity checks, and coefficient figures.

**Public Data Limitation:** Row-level source data and execution outputs are excluded; this notebook is intended for workflow review rather than full public rerun.


# Section 6: Statistical Modeling & Robustness Diagnostics

This section documents the county-clustered OLS modeling workflow used to characterize descriptive rating divergence. It also records diagnostic and sensitivity-analysis code used for the final report.

**Key Steps:**
1. Fit two baseline star-rating models and three percentile-gap models.
2. Use county-clustered robust standard errors.
3. Generate aggregate coefficient forest plots.
4. Run residual, heteroskedasticity, and VIF diagnostics.
5. Run Kaiser-exclusion and Cook's-distance sensitivity checks.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
from scipy.stats import shapiro
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:

# Public path constants. The row-level final analytic dataset is excluded from the public repository.
FINAL_ANALYTIC_DATASET_PATH = "data/excluded/final_analytic_dataset.csv"


In [ ]:

# Public GitHub output paths for aggregate artifacts.
PROJECT_ROOT = Path("..")
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


## 6.1 Load Dataset and Fit Regression Models

In [ ]:
df = pd.read_csv(FINAL_ANALYTIC_DATASET_PATH)

# Exclude missing value in each gaps
# Public copy: removed redundant overwritten df_clean_gap assignment.
df_clean_gap = df.dropna(subset=['gap_star_pct', 'SVI_idx_z', 'Gini_idx_z', 'log_google_reviews_z', 'flag_is_kaiser', 'flag_is_academic', 'cms_county'])

df_clean_clin = df.dropna(subset=['gap_google_clinical', 'SVI_idx_z', 'Gini_idx_z', 'log_google_reviews_z', 'flag_is_kaiser', 'flag_is_academic', 'cms_county'])
df_clean_exp = df.dropna(subset=['gap_google_official_exp', 'SVI_idx_z', 'Gini_idx_z', 'log_google_reviews_z', 'flag_is_kaiser', 'flag_is_academic', 'cms_county'])


In [ ]:

def run_clustered_model(target_col, data, title, features, cluster_col='cms_county'):
    """
    Clustered SE OLS Regression 
    using cov_type='cluster' to cluster samples at cluster_col column.
    """
    print(f"\n{'='*20} {title} {'='*20}")
    # get formula
    formula = f"{target_col} ~ {features}"
    
    # fit model
    model = smf.ols(formula, data=data)
    result = model.fit(cov_type='cluster', cov_kwds={'groups': data[cluster_col]})
    
    print(result.summary().tables[1])
    return result

# 1. Define all models: 
#    Title: (outcome variable, dataframe)

model_configs = {
    # Baseline model: using initial raw stars
    "[Baseline] Google Star (Raw)": ("google_star", df_clean_gap),
    "[Baseline] CMS Star (Raw)": ("cms_star", df_clean_gap),
    # Gap analysis model: using percentile rank ratings
    "[Gap 1] Google - CMS Star (Percentile Rank)": ("gap_star_pct", df_clean_gap),
    "[Gap 2] Google - CMS Patient Exp (Percentile Rank)": ("gap_google_official_exp", df_clean_exp),
    "[Gap 3] Google - CMS Clinical Outcome (Percentile Rank)": ("gap_google_clinical", df_clean_clin)
}

# 2. feature initialization
features = "SVI_idx_z + Gini_idx_z + log_google_reviews_z + flag_is_kaiser + flag_is_academic"

# 3. main models
main_results = {}
for title, (target, df_sub) in model_configs.items():
    main_results[title] = run_clustered_model(target, df_sub, title, features)


In [ ]:
# 4. Coeffient VIsualization

def plot_forest(results_dict, variables_to_plot, var_labels=None, color_map=None, fz = 12, title="Coefficient Forest Plot"):
    """
    Forest plot for coeffients confidence interval (CI) .
    """
    plot_data = []
    
    # 1. get results
    for model_name, res in results_dict.items():
        for var in variables_to_plot:
            if var in res.params:
                plot_data.append({
                    'Model': model_name,
                    'Original_Var': var,
                    'Variable': var_labels.get(var, var) if var_labels else var,
                    'Coef': res.params[var],
                    'CI_lower': res.conf_int().loc[var, 0],
                    'CI_upper': res.conf_int().loc[var, 1],
                    'P_value': res.pvalues[var]
                })
                
    df_plot = pd.DataFrame(plot_data)
    df_plot = df_plot.iloc[::-1].reset_index(drop=True)
    
    # 2. plot
    fig, ax = plt.subplots(figsize=(fz, len(df_plot) * 0.7))
    
    for i, row in df_plot.iterrows():
        # color for CI lines
        line_color = 'slategrey'
        if color_map and row['Original_Var'] in color_map:
            line_color = color_map[row['Original_Var']]
            
        # find significant coef
        is_sig = row['P_value'] < 0.05
        color = 'firebrick' if is_sig else 'slategrey'
        alpha = 1.0 if is_sig else 0.6
        
        # CI
        ax.errorbar(row['Coef'], i, 
                    xerr=[[row['Coef'] - row['CI_lower']], [row['CI_upper'] - row['Coef']]], 
                    fmt='o', color=line_color, ecolor=line_color, alpha=alpha, capsize=5, markersize=8)
        
        # significant level
        star = '*' if row['P_value'] < 0.05 else ''
        star += '*' if row['P_value'] < 0.01 else ''
        ax.text(row['CI_upper'] + 0.05, i, f"{row['Coef']:.2f}{star}", 
                va='center', color=color, fontsize=10, fontweight='bold' if is_sig else 'normal')

    # 3. axis and label setting
    ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_yticks(range(len(df_plot)))
    
    yticklabels = [f"{row['Model']}\n[{row['Variable']}]" for _, row in df_plot.iterrows()]
    ax.set_yticklabels(yticklabels, fontsize=10)
    
    ax.set_xlabel('Coefficient Estimate (95% CI)', fontsize=11, fontweight='bold')
    # ax.set_title(title, pad=20, fontsize=14, fontweight='bold')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    # 
    filename = title.replace(" ", "_").replace(":", "").lower()
    plt.savefig(FIGURE_DIR / f'forest_{filename}.png', bbox_inches='tight', transparent=True)
    plt.show()


# 1. SDOH forest plot
plot_forest(
    results_dict=main_results, 
    variables_to_plot=['SVI_idx_z'],
    var_labels={'SVI_idx_z': 'SVI'}, 
    fz=10, 
    title="Social Vulnerability Index (Poverty) across Models"
)

plot_forest(
    results_dict=main_results, 
    variables_to_plot=['Gini_idx_z'],
    var_labels={'Gini_idx_z': 'Gini index'},
    fz=10, 
    title="Gini Index (Inequality) across Models"
)

# 2. Google Review Counts forest plot
plot_forest(
    results_dict=main_results, 
    variables_to_plot=['log_google_reviews_z'],
    var_labels={'log_google_reviews_z': 'Google Review Counts'}, 
    fz=10, 
    title="Google Review Counts (Traffic) across Models"
)

# 3. Institutional subgroup marker forest plot
plot_forest(
    results_dict=main_results, 
    variables_to_plot=['flag_is_kaiser', 'flag_is_academic'],
    var_labels={'flag_is_kaiser': 'Kaiser / integrated HMO', 'flag_is_academic': 'Academic Medical Center'},
    color_map={'flag_is_kaiser': 'darkorange', 'flag_is_academic': 'steelblue'},
    title="Institutional subgroup markers across models"
)

## 6.2 Model Diagnostic

In [ ]:

def run_model_diagnostics(res, title="Model_Diagnostic"):
    """
    Run model diagnostic for OLS regression.
    Includes:
    1. Plots: Residuals vs Fitted plot & Normal Q-Q plot.
    2. Statistical Tests: Shapiro-Wilk (Normality) & Breusch-Pagan (Homoscedasticity).
    3. Multicollinearity Check: Variance Inflation Factor (VIF).
    """
    print(f"\n{'='*20} Diagnostics: {title} {'='*20}")
    
    # Extract embedded attributes directly from the result object
    residuals = res.resid
    fitted_vals = res.fittedvalues
    exog = res.model.exog
    exog_names = res.model.exog_names
    
    # ---------------------------------------------------------
    # Plot 1: Residuals vs Fitted 
    # ---------------------------------------------------------
    plt.figure(figsize=(8, 6))
    sns.residplot(x=fitted_vals, y=residuals, lowess=True, 
                  scatter_kws={'alpha': 0.6, 'edgecolor': 'black', 'color': 'slategrey'}, 
                  line_kws={'color': 'firebrick', 'lw': 2})
    plt.axhline(0, color='grey', linestyle='dashed')
    # plt.title(f'Residuals vs Fitted: {title}', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Fitted Values', fontsize=12)
    plt.ylabel('Residuals', fontsize=12)
    
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.tight_layout()
    
    safe_title = title.replace(" ", "_").replace("[", "").replace("]", "").lower()
    plt.savefig(FIGURE_DIR / f'diag_resid_{safe_title}.png', bbox_inches='tight', transparent=True)
    plt.show()

    # ---------------------------------------------------------
    # Plot 2: Normal Q-Q Plot 
    # ---------------------------------------------------------
    fig = sm.qqplot(residuals, line='45', fit=True, alpha=0.6, 
                    markerfacecolor='slategrey', markeredgecolor='black')
    # plt.title(f'Normal Q-Q Plot: {title}', fontsize=14, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f'diag_qq_{safe_title}.png', bbox_inches='tight', transparent=True)
    plt.show()

    # ---------------------------------------------------------
    # Statistical Tests & Output 
    # ---------------------------------------------------------
    # 1. Shapiro-Wilk Test
    shapiro_stat, shapiro_p = stats.shapiro(residuals)
    normality_msg = "Pass (Perfect Normality)" if shapiro_p > 0.05 else "Fail (Robust inference valid via CLT given N>200)"
    print(f"1. Shapiro-Wilk Test (Normality):")
    print(f"   p-value = {shapiro_p:.4f} -> [{normality_msg}]\n")

    # 2. Breusch-Pagan Test
    bp_stat, bp_p, f_stat, f_p = het_breuschpagan(residuals, exog)
    bp_msg = "Pass (Homoscedastic)" if bp_p > 0.05 else "Fail (Heteroscedasticity addressed by Clustered SE)"
    print(f"2. Breusch-Pagan Test (Homoscedasticity):")
    print(f"   p-value = {bp_p:.4f} -> [{bp_msg}]\n")

    # 3. VIF Calculation
    vif_data = pd.DataFrame()
    vif_data["Feature"] = exog_names
    vif_data["VIF"] = [variance_inflation_factor(exog, i) for i in range(exog.shape[1])]
    
    # Filter out intercept for max VIF check
    features_only = vif_data[vif_data["Feature"] != "Intercept"]
    max_vif = features_only["VIF"].max()
    vif_msg = "Pass (No Multicollinearity)" if max_vif < 5 else "Warning (High Multicollinearity detected)"
    
    print(f"3. Variance Inflation Factor (VIF):")
    print(f"   Max VIF = {max_vif:.4f} -> [{vif_msg}]")
    print("-" * 40)
    print(vif_data.to_string(index=False))

for title, res in main_results.items():
    run_model_diagnostics(res, title)

## 6.3 Sensitivity Analysis

In [ ]:
# 1. sensitivity analysis without kaiser samples
print("-"*20 + "Sensitivity Analysis 1: No Kaiser" + "-"*20)
# initialization
sens_results = {}
sens_features = "SVI_idx_z + Gini_idx_z + log_google_reviews_z + flag_is_academic" # No kaiser for sensitivity

for title, (target, df_sub) in model_configs.items():
    # drop kaiser samples
    df_sub_no_kaiser = df_sub[df_sub['flag_is_kaiser'] == 0]
    sens_results[title] = run_clustered_model(target, df_sub_no_kaiser, f"[Sensitivity 1] {title}", sens_features)

In [ ]:
# 2. Diagnostic via Cook's Distance
# 3. Sensitivity analysis without influential samples

def detect_and_remove_outliers(model_res, data, title, threshold_multiplier=4):
    """
    Using Cook's distance to find outliers.
    """
    # cook's distance
    influence = model_res.get_influence()
    cooks_d = influence.cooks_distance[0]
    
    # threshold = 4/n
    n_obs = len(data)
    threshold = threshold_multiplier / n_obs
    
    # find outliers
    outlier_mask = cooks_d > threshold
    outliers_count = outlier_mask.sum()
    
    print(f"[{title}] N={n_obs} | Threshold={threshold:.4f} | Outliers Removed: {outliers_count}")
    
    # drop outliers and return dataframe for sensitivity analysis 2
    return data[~outlier_mask].copy()

# sensitivity analysis without influential samples
print("-"*20 + "Sensitivity Analysis 2: No Influential Samples" + "-"*20)

# initialization
robust_results = {}

for title, (target, df_sub) in model_configs.items():
    # 1. get formula from previous main model
    baseline_res = main_results[title]
    
    # 2. drop outliers
    df_robust = detect_and_remove_outliers(baseline_res, df_sub, title)
    
    # 3. sensitivity analysis: run Clustered SE OLS again
    robust_results[title] = run_clustered_model(
        target_col=target, 
        data=df_robust, 
        title=f"[Sensitivity 2] {title}", 
        features=features
    )